# Session 4 — Evaluate Faithfulness with GLIDE

In Session 2 you built an agent that answers questions from NVIDIA's filings. Before relying
on it, you need to know how often its answers are **faithful**, that is, supported by the
passages it read, with nothing invented or distorted.

Measuring this is harder than it looks. Checking every answer by hand is reliable, but far too
slow and expensive to do at scale. An LLM can check thousands of answers in minutes, but it makes
mistakes of its own, and those mistakes tend to push its figure in one direction.

This session combines the two in two steps:

1. An LLM **judges** every claim.
2. A few human labels **correct** the judge's bias, thanks to GLIDE.

We work on claims about NVIDIA whose true labels are known, so every estimate can be checked
against the truth.

## Setup

The helpers used in this notebook are given to you in `utils/evaluate.py`. Open it and have a
look before going further.

In [ ]:
import json
import random
from pathlib import Path

from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage

from utils.evaluate import judge_all, save_verdicts

CLAIMS_PATH = Path("../data/06_claims/paragraph_claims.json")

claims = json.loads(CLAIMS_PATH.read_text())
print(len(claims), "claims")

## The claims

Each claim is a short sentence about one chunk of NVIDIA's filings. We wrote them before the course.  Half of them are
**faithful**: everything they say can be deduced from their chunk. The other half are
**unfaithful**: they distort one fact of the chunk, often subtly. We wrote them that way on
purpose, which is why their true label is known.

A row has four fields that matter here:

- `claim` is the sentence to judge.
- `chunk` is the passage it should be deduced from.
- `true_faithfulness_label` is 1 for a faithful claim and 0 for an unfaithful one.
- `error_type` names the kind of distortion in an unfaithful claim, such as a reversal or an
  exaggeration.

In [ ]:
from IPython.display import Markdown

rows = [f"| {row['claim']} | {row['true_faithfulness_label']} | {row['error_type'] or ''} |" for row in claims[:6]]
Markdown("| claim | label | error type |\n|---|---|---|\n" + "\n".join(rows))

Every chunk has one faithful and one unfaithful claim. Here are the two claims written from the
same chunk, below the chunk itself:

In [ ]:
pair = [row for row in claims if row["chunk_id"] == "10K-NVDA-2025_c0114"]

print(pair[0]["chunk"], "\n")
for row in pair:
    print(f"[{row['true_faithfulness_label']}] {row['claim']}")

## The judge

> **⚠️ This part calls the API with the key you set up in Session 2**, so there is nothing to
> change. The key is shared by the whole room: please use it sparingly.

The judge is the same small model as in Session 2. Each call sends it the judge's instructions
and one claim, and gets back one verdict.

In [ ]:
load_dotenv()

llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0.0, max_tokens=200, max_retries=5)

print("model ready:", llm.model)

### The judge's prompt

The prompt below is given to you. It follows these rules:

- A claim is faithful only if **every** piece of information in it can be deduced from the
  chunk. One unsupported detail is enough to reject it.
- The judge must use the chunk alone, not what it knows about NVIDIA.
- It writes its reasoning **before** its verdict, so that it reasons first instead of
  justifying a verdict it has already chosen.
- It gets no examples, because examples tend to bias a judge in unpredictable directions.
- It answers in JSON, so that code can read the verdict.

In [ ]:
JUDGE_PROMPT = """You are a judge evaluating the faithfulness of claims produced by a financial RAG system. You will receive a chunk of text extracted from an NVIDIA 10-K filing or earnings-call transcript, and one claim about it. In these chunks, "we" and "our" refer to NVIDIA.

Your task is to decide whether the claim is deducible from the chunk. A claim is deducible only if every piece of information it contains follows from the chunk. As soon as one piece of information cannot be deduced from the chunk, the whole claim is not deducible.

Proceed as follows:

1. Read the chunk in full.
2. Read the claim in full.
3. Check whether each piece of information in the claim follows from the chunk, and write this reasoning down in under 100 words.
4. Only then, give the verdict that follows from your reasoning.

Judge the claim against the chunk alone, without outside knowledge: a claim that is true in the world but not supported by the chunk is not deducible. The chunk was cut at a fixed length, so it may start or end mid-sentence or mix table rows with prose.

The verdict is a label:

- 1: every piece of information in the claim is deducible from the chunk.
- 0: at least one piece of information is not deducible from the chunk.

Return only a JSON object, with no text before or after it:

{"reasoning": "<your reasoning, in under 100 words>", "verdict": <0 or 1>}
"""

## Exercise 1 — Judging one claim

A judge is only useful if it can be called on thousands of claims without anyone reading its
answers. Every call must therefore send the same kind of request, and its reply must be read back
as data rather than as text.

**Objective:** Ask the judge whether one claim is faithful to its chunk, and return its judgement
as a dict.

**Your task:** Complete `score_faithfulness` below, respecting its signature and the intent
documented in its docstring.

In [ ]:
def score_faithfulness(chunk: str, claim: str, llm: ChatAnthropic, system_prompt: str) -> dict:
    """Ask an LLM judge whether a claim is deducible from a chunk.

    Parameters
    ----------
    chunk : str
        The text the claim should be deducible from.
    claim : str
        The claim to judge.
    llm : ChatAnthropic
        The judge model.
    system_prompt : str
        The judge's instructions, which ask for a JSON object with a reasoning and a verdict.

    Returns
    -------
    dict
        The judgement: "reasoning", its explanation, and "verdict", 1 if the claim is deducible
        from the chunk and 0 if not.
    """
    # STEP 1 - Build the two messages: the instructions, then the chunk and the claim in an f-string.
    messages = ...  # YOUR CODE HERE

    # STEP 2 - Send them to the judge.
    response = ...  # YOUR CODE HERE

    # A reply cut off by max_tokens is incomplete JSON, so fail loudly rather than misread it.
    if response.response_metadata.get("stop_reason") == "max_tokens":
        raise ValueError(f"the judge's reply was truncated: {response.content!r}")

    # STEP 3 - Read the JSON reply into a dict.
    return ...  # YOUR CODE HERE

**Hint:** Build the messages as `answer_question` did in Session 2, with `SystemMessage` and
`HumanMessage`, and write the second one as `f"Chunk:\n{chunk}\n\nClaim:\n{claim}"`.
`llm.invoke` sends a list of messages to the model, and `json.loads` reads the text of the reply,
`response.text`.

### Check your implementation

If an assertion fails, your function is not correct yet. The first two assertions check the
messages you send (the prompt as a `SystemMessage`, then a `HumanMessage`), the third checks that
the chunk comes before the claim, and the last checks that you return the dict read from the
reply.

In [ ]:
from langchain_core.messages import AIMessage


class FakeJudge:
    """Stands in for the model: records the messages it receives and gives a fixed answer."""

    def invoke(self, messages):
        self.messages = messages
        return AIMessage('{"reasoning": "test", "verdict": 1}', response_metadata={"stop_reason": "end_turn"})


fake_judge = FakeJudge()
judgement = score_faithfulness("THE CHUNK", "THE CLAIM", fake_judge, JUDGE_PROMPT)

system_message, human_message = fake_judge.messages
assert isinstance(system_message, SystemMessage) and system_message.content == JUDGE_PROMPT
assert isinstance(human_message, HumanMessage)
assert human_message.content.index("THE CHUNK") < human_message.content.index("THE CLAIM")
assert judgement == {"reasoning": "test", "verdict": 1}
print("OK: the judge receives the prompt, then the chunk and the claim")

## Exercise 2 — Measuring the judge

A single accuracy figure can hide where a judge goes wrong. A judge that rejects true claims and a
judge that accepts false ones can have the same overall accuracy, yet they push the faithfulness
rate in opposite directions. Measuring accuracy separately on faithful and unfaithful claims shows
which of the two you have.

**Objective:** Measure how often the judge is right on faithful claims, on unfaithful claims and
overall.

**Your task:** Complete `accuracy_per_class` below, respecting its signature and the intent
documented in its docstring.

In [ ]:
def accuracy_per_class(verdicts: list[int | None], labels: list[int]) -> dict[str, float]:
    """Measure how often the judge is right, on faithful claims, on unfaithful claims and overall.

    Parameters
    ----------
    verdicts : list[int | None]
        The judge's verdict for each claim, 1 or 0, or None where the call failed.
    labels : list[int]
        The true label of each claim, in the same order: 1 if faithful, 0 if not.

    Returns
    -------
    dict[str, float]
        The share of claims judged correctly under "faithful", "unfaithful" and "overall".
        Claims without a verdict are left out.
    """
    # STEP 1 - Pair each verdict with its label, leaving out the claims without a verdict.
    judged = ...  # YOUR CODE HERE

    # STEP 2 - Compute the share of pairs whose verdict equals their label.
    def compute_accuracy(pairs: list[tuple[int, int]]) -> float:
        return ...  # YOUR CODE HERE

    # The same share on faithful claims, on unfaithful claims and on all of them.
    return {
        "faithful": compute_accuracy([pair for pair in judged if pair[1] == 1]),
        "unfaithful": compute_accuracy([pair for pair in judged if pair[1] == 0]),
        "overall": compute_accuracy(judged),
    }

**Hint:** `zip` pairs two lists element by element, and an `if` at the end of a list
comprehension filters it. To count the correct pairs, `sum` works on booleans, since `True`
counts as 1.

### Check your implementation

In [ ]:
accuracy = accuracy_per_class([1, 1, 0, None, 1, 0], [1, 1, 1, 0, 0, 0])

assert accuracy == {"faithful": 2 / 3, "unfaithful": 1 / 2, "overall": 3 / 5}
print("OK:", accuracy)

### Try the judge on a few claims

The cell below runs your judge on 5 faithful and 5 unfaithful claims. Run it before judging the
whole dataset, to catch a bug in 10 calls rather than 800.

In [ ]:
rng = random.Random(19)
sample = rng.sample([row for row in claims if row["true_faithfulness_label"] == 1], 5)
sample += rng.sample([row for row in claims if row["true_faithfulness_label"] == 0], 5)

judgements = [score_faithfulness(row["chunk"], row["claim"], llm, JUDGE_PROMPT) for row in sample]

assert all(set(judgement) == {"reasoning", "verdict"} for judgement in judgements)
assert all(judgement["verdict"] in (0, 1) for judgement in judgements)

accuracy = accuracy_per_class(
    [judgement["verdict"] for judgement in judgements], [row["true_faithfulness_label"] for row in sample]
)
for name, value in accuracy.items():
    print(f"{name}: {value:.1%}")

### Judge every claim

`judge_all` runs your function on every claim, eight calls at a time, and returns one verdict
per claim. It takes about three minutes, so run it once. A call that fails gives `None` instead
of a verdict.

In [ ]:
verdicts = judge_all(score_faithfulness, claims, llm, JUDGE_PROMPT)

print(sum(verdict is not None for verdict in verdicts), "claims judged,", verdicts.count(None), "failed")

### Save the verdicts

The next part reads the verdicts from this file, so you will not have to judge the claims again
after a restart.

In [ ]:
VERDICTS_PATH = Path("../data/07_verdicts/verdicts.json")
VERDICTS_PATH.parent.mkdir(parents=True, exist_ok=True)

save_verdicts(verdicts, [row["claim_id"] for row in claims], str(VERDICTS_PATH))
print(f"{len(verdicts)} verdicts in {VERDICTS_PATH}")

### How good is the judge?

Here we are lucky: every claim has a known label, so the judge can be measured on all of them.

In [ ]:
accuracy = accuracy_per_class(verdicts, [row["true_faithfulness_label"] for row in claims])

for name, value in accuracy.items():
    print(f"{name}: {value:.1%}")

The judge is right about 95% of the time, which looks excellent. The split tells a different
story. It almost never rejects a faithful claim (99.8%), but it lets about one unfaithful claim in
ten through (90%). Its errors all point the same way, so it overstates how faithful the claims
are.

In a real project, nobody has the labels of every claim, so this bias would stay invisible. The
next part shows how a few human labels are enough to measure it and correct it.